# Replicação parcial do PsyAttention: BigFive Essays via Hugging Face

Este notebook implementa a **primeira etapa de uma replicação parcial** do estudo PsyAttention usando o dataset Big Five Essays carregado diretamente do Hugging Face:

```python
from datasets import load_dataset
ds = load_dataset("jingjietan/essays-big5")
```

O objetivo deste notebook é deixar a pipeline de dados e o baseline principal prontos:

1. carregar o dataset pelo Hugging Face;
2. criar split fixo 70/20/10;
3. normalizar labels dos cinco traços Big Five;
4. rodar baselines rápidos;
5. treinar um `bert-base-uncased` para cada traço de personalidade;
6. salvar resultados, splits e relatórios.

> Observação: este notebook cobre a etapa inicial da replicação. A parte completa de PsyAttention com SEANCE/TAACO/TAALES e encoder psicológico deve vir depois que esta pipeline estiver validada.


## 0. Instalação

Execute esta célula se estiver em um ambiente novo, como Colab, Kaggle ou uma VM limpa.


In [1]:
# Descomente se necessário.
# %pip install -q datasets transformers accelerate evaluate scikit-learn pandas numpy torch tqdm

## 1. Imports e configuração

Use `MODE = "debug"` para testar a pipeline rapidamente.  
Use `MODE = "replication"` para uma execução mais próxima da configuração do artigo.


In [2]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, concatenate_datasets, DatasetDict
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# -----------------------------
# Configuração principal
# -----------------------------

DATASET_NAME = "jingjietan/essays-big5"
MODEL_NAME = "bert-base-uncased"

# Modos sugeridos:
# "debug"          -> teste rápido, só para ver se roda
# "replication_3"  -> dataset inteiro, 3 épocas
# "replication_5"  -> dataset inteiro, 5 épocas, bom próximo teste
# "replication_10" -> dataset inteiro, 10 épocas, teste mais longo
# "paper_like"     -> 50 épocas, mais próximo do artigo, mas deixe para depois
MODE = "replication_10"

SEED = 42

OUTPUT_DIR = Path("./outputs_essays_big5_hf")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# No Hugging Face, max_length inclui [CLS] e [SEP].
# Então 512 equivale a aproximadamente 510 tokens reais de texto.
MAX_LENGTH = 512

# Traços Big Five que serão treinados.
RUN_TRAITS = ["OPN", "CON", "EXT", "AGR", "NEU"]

# -----------------------------
# Configuração por modo
# -----------------------------

if MODE == "debug":
    # Execução curta para testar se tudo está funcionando.
    NUM_EPOCHS = 1
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8

    MAX_TRAIN_SAMPLES = 200
    MAX_VALID_SAMPLES = 100
    MAX_TEST_SAMPLES = 100

elif MODE == "replication_3":
    # Primeiro run sério com dataset inteiro.
    NUM_EPOCHS = 3
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

    MAX_TRAIN_SAMPLES = None
    MAX_VALID_SAMPLES = None
    MAX_TEST_SAMPLES = None

elif MODE == "replication_5":
    # Minha recomendação para o próximo run com GPU forte.
    NUM_EPOCHS = 5
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

    MAX_TRAIN_SAMPLES = None
    MAX_VALID_SAMPLES = None
    MAX_TEST_SAMPLES = None

elif MODE == "replication_10":
    # Teste para ver se o BERT melhora ou começa a overfitar.
    NUM_EPOCHS = 10
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

    MAX_TRAIN_SAMPLES = None
    MAX_VALID_SAMPLES = None
    MAX_TEST_SAMPLES = None

elif MODE == "paper_like":
    # Mais próximo do artigo, mas eu só rodaria depois de validar 3/5/10 épocas.
    NUM_EPOCHS = 50
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

    MAX_TRAIN_SAMPLES = None
    MAX_VALID_SAMPLES = None
    MAX_TEST_SAMPLES = None

else:
    raise ValueError(f"MODE inválido: {MODE}")

# -----------------------------
# Hiperparâmetros principais
# -----------------------------

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Para experimentos futuros.
# Seu loop atual provavelmente ainda usa só SEED.
# Depois podemos adaptar o notebook para rodar todos automaticamente.
SEEDS_TO_TEST = [13, 42, 100]
LEARNING_RATES_TO_TEST = [1e-5, 2e-5, 3e-5]

# -----------------------------
# Configurações de reprodutibilidade
# -----------------------------

os.environ["TOKENIZERS_PARALLELISM"] = "false"

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

    # Acelera treino em GPUs modernas, especialmente Ampere/Ada/Hopper.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# -----------------------------
# Informações do ambiente
# -----------------------------

print("Modo:", MODE)
print("Dataset:", DATASET_NAME)
print("Modelo:", MODEL_NAME)
print("Seed:", SEED)
print("Épocas:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Max length:", MAX_LENGTH)
print("Batch train:", TRAIN_BATCH_SIZE)
print("Batch eval:", EVAL_BATCH_SIZE)
print("Run traits:", RUN_TRAITS)
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memória GPU total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Modo: replication_10
Dataset: jingjietan/essays-big5
Modelo: bert-base-uncased
Seed: 42
Épocas: 10
Learning rate: 2e-05
Max length: 512
Batch train: 8
Batch eval: 16
Run traits: ['OPN', 'CON', 'EXT', 'AGR', 'NEU']
GPU disponível: True
GPU: NVIDIA L4
Memória GPU total GB: 23.66


## 2. Carregar o dataset do Hugging Face

Esta é a mudança principal: em vez de ler um CSV local, o notebook baixa o dataset diretamente com `load_dataset`.


In [3]:
raw_ds = load_dataset(DATASET_NAME)

print(raw_ds)
print("Splits disponíveis:", list(raw_ds.keys()))

# Alguns datasets no Hugging Face vêm só com split "train".
# Para manter o controle do split 70/20/10, juntamos todos os splits existentes antes de criar os nossos.
if len(raw_ds.keys()) == 1:
    all_ds = raw_ds[list(raw_ds.keys())[0]]
else:
    all_ds = concatenate_datasets([raw_ds[split] for split in raw_ds.keys()])

# Guarda um ID estável da linha original para salvar os splits.
if "row_id" not in all_ds.column_names:
    all_ds = all_ds.add_column("row_id", list(range(len(all_ds))))

print("Total de exemplos:", len(all_ds))
print("Colunas:", all_ds.column_names)
print(all_ds[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/795k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/953k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1578 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/395 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/494 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 1578
    })
    validation: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 395
    })
    test: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 494
    })
})
Splits disponíveis: ['train', 'validation', 'test']
Total de exemplos: 2467
Colunas: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__', 'row_id']
{'O': '1', 'C': '0', 'E': '0', 'A': '1', 'N': '1', 'ptype': '19', 'text': "it is wednesday. I can't wait until friday because I am going home to see brandon. I miss him so much. I can't wait to see him. two more days. this has been a very long two weeks. time passes very slowly here. I have a lot of free time on my hands when I am not in class. class. psychology class. psychology is fun so far. it really interests me

## 3. Detectar coluna de texto e colunas de labels

O dataset pode vir com nomes de coluna diferentes dependendo da versão.  
Esta célula tenta detectar os nomes mais comuns, como `text`, `TEXT`, `cOPN`, `cCON`, `cEXT`, `cAGR`, `cNEU`.


In [4]:
def find_first_existing_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

columns = all_ds.column_names

TEXT_COL = find_first_existing_column(
    columns,
    ["text", "TEXT", "essay", "Essay", "content", "document"]
)

if TEXT_COL is None:
    raise ValueError(
        f"Não consegui detectar a coluna de texto. Colunas disponíveis: {columns}"
    )

# Mapeia nomes canônicos dos traços para possíveis nomes de coluna no dataset.
LABEL_CANDIDATES = {
    "OPN": ["O", "cOPN", "OPN", "opn", "openness", "Openness", "OPENNESS"],
    "CON": ["C", "cCON", "CON", "con", "conscientiousness", "Conscientiousness", "CONSCIENTIOUSNESS"],
    "EXT": ["E", "cEXT", "EXT", "ext", "extraversion", "Extraversion", "EXTRAVERSION"],
    "AGR": ["A", "cAGR", "AGR", "agr", "agreeableness", "Agreeableness", "AGREEABLENESS"],
    "NEU": ["N", "cNEU", "NEU", "neu", "neuroticism", "Neuroticism", "NEUROTICISM"],
}

LABEL_COLS = {}
for trait, candidates in LABEL_CANDIDATES.items():
    col = find_first_existing_column(columns, candidates)
    if col is not None:
        LABEL_COLS[trait] = col

if len(LABEL_COLS) != 5:
    raise ValueError(
        "Não consegui detectar todas as cinco colunas Big Five.\n"
        f"Detectadas: {LABEL_COLS}\n"
        f"Colunas disponíveis: {columns}"
    )

print("Coluna de texto:", TEXT_COL)
print("Colunas de labels:", LABEL_COLS)


Coluna de texto: text
Colunas de labels: {'OPN': 'O', 'CON': 'C', 'EXT': 'E', 'AGR': 'A', 'NEU': 'N'}


## 4. Normalizar labels binárias

Os labels podem vir como `y/n`, `true/false`, `0/1`, booleanos ou inteiros.  
Esta célula converte tudo para `0` ou `1`.


In [5]:
def normalize_binary_label(value):
    if value is None:
        return None

    if isinstance(value, (bool, np.bool_)):
        return int(value)

    if isinstance(value, (int, np.integer)):
        if value in [0, 1]:
            return int(value)
        raise ValueError(f"Label inteiro inesperado: {value}")

    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return None
        if value in [0.0, 1.0]:
            return int(value)
        raise ValueError(f"Label float inesperado: {value}")

    if isinstance(value, str):
        v = value.strip().lower()
        positives = {"1", "1.0", "true", "t", "yes", "y", "positive", "pos", "high"}
        negatives = {"0", "0.0", "false", "f", "no", "n", "negative", "neg", "low"}
        if v in positives:
            return 1
        if v in negatives:
            return 0
        raise ValueError(f"Label string inesperado: {value!r}")

    raise TypeError(f"Tipo de label inesperado: {type(value)} | valor={value!r}")


def normalize_example(example):
    for trait, col in LABEL_COLS.items():
        example[col] = normalize_binary_label(example[col])
    return example


all_ds = all_ds.map(normalize_example)

# Remove exemplos com texto vazio ou labels ausentes.
def is_valid_example(example):
    if example[TEXT_COL] is None or str(example[TEXT_COL]).strip() == "":
        return False
    for col in LABEL_COLS.values():
        if example[col] is None:
            return False
    return True

all_ds = all_ds.filter(is_valid_example)

print("Total após limpeza:", len(all_ds))

# Conferência rápida das distribuições de classe.
label_summary = {}
for trait, col in LABEL_COLS.items():
    values = pd.Series(all_ds[col])
    label_summary[trait] = values.value_counts().sort_index().to_dict()

pd.DataFrame(label_summary).T.rename(columns={0: "classe_0", 1: "classe_1"})


Map:   0%|          | 0/2467 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2467 [00:00<?, ? examples/s]

Total após limpeza: 2467


,classe_0,classe_1
OPN,1196,1271
CON,1214,1253
EXT,1191,1276
AGR,1157,1310
NEU,1234,1233


## 5. Criar split fixo 70/20/10

O artigo usa uma divisão 7:2:1.  
Aqui criamos:

- 70% treino
- 20% validação
- 10% teste

Também salvamos os `row_id` de cada split para permitir repetição posterior.


In [6]:
train_test = all_ds.train_test_split(
    test_size=0.30,
    seed=SEED,
    shuffle=True,
)

valid_test = train_test["test"].train_test_split(
    test_size=1/3,  # 1/3 de 30% = 10% total
    seed=SEED,
    shuffle=True,
)

ds = DatasetDict({
    "train": train_test["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

print(ds)
print("Proporções:")
print({
    split: round(len(ds[split]) / len(all_ds), 3)
    for split in ds.keys()
})

split_ids = {
    split: list(ds[split]["row_id"])
    for split in ds.keys()
}

with open(OUTPUT_DIR / "split_row_ids.json", "w", encoding="utf-8") as f:
    json.dump(split_ids, f, ensure_ascii=False, indent=2)

print("IDs dos splits salvos em:", OUTPUT_DIR / "split_row_ids.json")


DatasetDict({
    train: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__', 'row_id'],
        num_rows: 1726
    })
    validation: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__', 'row_id'],
        num_rows: 494
    })
    test: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__', 'row_id'],
        num_rows: 247
    })
})
Proporções:
{'train': 0.7, 'validation': 0.2, 'test': 0.1}
IDs dos splits salvos em: outputs_essays_big5_hf/split_row_ids.json


## 6. Baseline 1: classe majoritária

Antes de usar BERT, é útil verificar a dificuldade mínima da tarefa.  
Este baseline sempre prevê a classe mais frequente no treino.


In [7]:
majority_results = []

for trait, label_col in LABEL_COLS.items():
    clf = DummyClassifier(strategy="most_frequent")

    y_train = np.array(ds["train"][label_col])
    y_test = np.array(ds["test"][label_col])

    # DummyClassifier espera X, mas ignora o conteúdo.
    clf.fit(np.zeros((len(y_train), 1)), y_train)
    preds = clf.predict(np.zeros((len(y_test), 1)))

    acc = accuracy_score(y_test, preds)
    majority_results.append({
        "model": "majority",
        "trait": trait,
        "label_col": label_col,
        "accuracy": acc,
    })

majority_df = pd.DataFrame(majority_results)
majority_df


,model,trait,label_col,accuracy
0,majority,OPN,O,0.582996
1,majority,CON,C,0.461538
2,majority,EXT,E,0.578947
3,majority,AGR,A,0.558704
4,majority,NEU,N,0.522267


## 7. Baseline 2: TF-IDF + Regressão Logística

Este baseline é rápido e ajuda a identificar problemas de label, split ou texto antes de treinar BERT.


In [8]:
tfidf_results = []

for trait, label_col in LABEL_COLS.items():
    vectorizer = TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        lowercase=True,
        min_df=2,
    )

    X_train = vectorizer.fit_transform(ds["train"][TEXT_COL])
    X_test = vectorizer.transform(ds["test"][TEXT_COL])

    y_train = np.array(ds["train"][label_col])
    y_test = np.array(ds["test"][label_col])

    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=None,
        random_state=SEED,
    )

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    tfidf_results.append({
        "model": "tfidf_logreg",
        "trait": trait,
        "label_col": label_col,
        "accuracy": acc,
    })

tfidf_df = pd.DataFrame(tfidf_results)
tfidf_df


,model,trait,label_col,accuracy
0,tfidf_logreg,OPN,O,0.578947
1,tfidf_logreg,CON,C,0.574899
2,tfidf_logreg,EXT,E,0.566802
3,tfidf_logreg,AGR,A,0.538462
4,tfidf_logreg,NEU,N,0.570850


## 8. Preparar BERT

Treinaremos um classificador binário separado para cada traço:

- OPN
- CON
- EXT
- AGR
- NEU


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def maybe_select(dataset, max_samples):
    if max_samples is None:
        return dataset
    return dataset.select(range(min(len(dataset), max_samples)))


def tokenize_for_trait(dataset_dict, label_col):
    def tokenize_batch(batch):
        encoded = tokenizer(
            batch[TEXT_COL],
            truncation=True,
            max_length=MAX_LENGTH,
        )
        encoded["labels"] = [int(x) for x in batch[label_col]]
        return encoded

    tokenized = DatasetDict()
    for split in ["train", "validation", "test"]:
        split_ds = dataset_dict[split]
        if split == "train":
            split_ds = maybe_select(split_ds, MAX_TRAIN_SAMPLES)
        elif split == "validation":
            split_ds = maybe_select(split_ds, MAX_VALID_SAMPLES)
        elif split == "test":
            split_ds = maybe_select(split_ds, MAX_TEST_SAMPLES)

        tokenized[split] = split_ds.map(
            tokenize_batch,
            batched=True,
            remove_columns=split_ds.column_names,
        )

    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds)
    }


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 9. Treinar BERT-base + MLP para cada traço

No `MODE = "debug"`, esta etapa usa poucos exemplos só para testar o notebook.  
Para rodar a replicação de verdade, altere `MODE = "replication"` na configuração inicial.


In [10]:
bert_results = []
bert_reports = {}

for trait, label_col in LABEL_COLS.items():
    print("\n" + "=" * 80)
    print(f"Treinando BERT para {trait} | coluna={label_col}")
    print("=" * 80)

    trait_output_dir = OUTPUT_DIR / f"bert_{trait.lower()}"
    trait_output_dir.mkdir(parents=True, exist_ok=True)

    tokenized = tokenize_for_trait(ds, label_col)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )

    training_args = TrainingArguments(
        output_dir=str(trait_output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        save_total_limit=1,
        seed=SEED,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    pred_output = trainer.predict(tokenized["test"])
    test_logits = pred_output.predictions
    test_labels = pred_output.label_ids
    test_preds = np.argmax(test_logits, axis=-1)

    test_acc = accuracy_score(test_labels, test_preds)
    report = classification_report(
        test_labels,
        test_preds,
        output_dict=True,
        zero_division=0,
    )

    bert_results.append({
        "model": "bert_base_mlp",
        "trait": trait,
        "label_col": label_col,
        "accuracy": test_acc,
        "mode": MODE,
        "epochs": NUM_EPOCHS,
        "max_train_samples": MAX_TRAIN_SAMPLES,
    })

    bert_reports[trait] = report

    trainer.save_model(str(trait_output_dir / "best_model"))

bert_df = pd.DataFrame(bert_results)
bert_df



Treinando BERT para OPN | coluna=O


Map:   0%|          | 0/1726 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,model,trait,label_col,accuracy,mode,epochs,max_train_samples
0,bert_base_mlp,OPN,O,0.582996,replication_10,10,None
1,bert_base_mlp,CON,C,0.578947,replication_10,10,None
2,bert_base_mlp,EXT,E,0.591093,replication_10,10,None
3,bert_base_mlp,AGR,A,0.599190,replication_10,10,None
4,bert_base_mlp,NEU,N,0.562753,replication_10,10,None


## 10. Consolidar resultados

A tabela abaixo combina:

- classe majoritária;
- TF-IDF + regressão logística;
- BERT-base + MLP.


In [11]:
all_results = pd.concat(
    [majority_df, tfidf_df, bert_df],
    ignore_index=True,
)

summary = (
    all_results
    .groupby("model", as_index=False)
    .agg(avg_accuracy=("accuracy", "mean"))
    .sort_values("avg_accuracy", ascending=False)
)

display(all_results)
display(summary)

all_results.to_csv(OUTPUT_DIR / "results_by_trait.csv", index=False)
summary.to_csv(OUTPUT_DIR / "results_summary.csv", index=False)

with open(OUTPUT_DIR / "bert_classification_reports.json", "w", encoding="utf-8") as f:
    json.dump(bert_reports, f, ensure_ascii=False, indent=2)

print("Resultados salvos em:")
print("-", OUTPUT_DIR / "results_by_trait.csv")
print("-", OUTPUT_DIR / "results_summary.csv")
print("-", OUTPUT_DIR / "bert_classification_reports.json")


,model,trait,label_col,accuracy,mode,epochs,max_train_samples
0,majority,OPN,O,0.582996,NaN,NaN,NaN
1,majority,CON,C,0.461538,NaN,NaN,NaN
2,majority,EXT,E,0.578947,NaN,NaN,NaN
3,majority,AGR,A,0.558704,NaN,NaN,NaN
4,majority,NEU,N,0.522267,NaN,NaN,NaN
5,tfidf_logreg,OPN,O,0.578947,NaN,NaN,NaN
6,tfidf_logreg,CON,C,0.574899,NaN,NaN,NaN
7,tfidf_logreg,EXT,E,0.566802,NaN,NaN,NaN
8,tfidf_logreg,AGR,A,0.538462,NaN,NaN,NaN
9,tfidf_logreg,NEU,N,0.570850,NaN,NaN,NaN


,model,avg_accuracy
0,bert_base_mlp,0.582996
2,tfidf_logreg,0.565992
1,majority,0.540891


Resultados salvos em:
- outputs_essays_big5_hf/results_by_trait.csv
- outputs_essays_big5_hf/results_summary.csv
- outputs_essays_big5_hf/bert_classification_reports.json


## 11. Comparação com os alvos do artigo

O artigo reporta, para BigFive Essays, uma acurácia média de aproximadamente:

- `BERT-base + MLP`: 60.6%
- `PsyAttention`: 65.66%

Este notebook, nesta primeira etapa, compara seus resultados locais contra o baseline BERT.  
Não espere bater exatamente os números do artigo se você estiver em `MODE = "debug"`.


In [12]:
TARGETS = pd.DataFrame([
    {"model": "paper_bert_base_mlp", "avg_accuracy": 0.606},
    {"model": "paper_psyattention", "avg_accuracy": 0.6566},
])

comparison = pd.concat(
    [
        summary[["model", "avg_accuracy"]],
        TARGETS,
    ],
    ignore_index=True,
)

comparison


,model,avg_accuracy
0,bert_base_mlp,0.582996
1,tfidf_logreg,0.565992
2,majority,0.540891
3,paper_bert_base_mlp,0.606000
4,paper_psyattention,0.656600


## 12. Próxima etapa: conectar recursos psicológicos

Depois que o baseline com Hugging Face estiver funcionando, a próxima parte da replicação é extrair os recursos psicológicos com SEANCE, TAACO e TAALES.

A célula abaixo deixa um ponto de entrada para uma matriz de features já extraída.  
Ela espera um CSV com:

- `row_id`
- uma coluna para cada feature psicológica

Assim, você mantém o mesmo split usado neste notebook.


In [13]:
# Opcional: use quando você já tiver extraído features psicológicas fora deste notebook.
PSY_FEATURES_CSV = None  # exemplo: "./features/essays_big5_psyfeatures.csv"

if PSY_FEATURES_CSV is not None:
    psy_df = pd.read_csv(PSY_FEATURES_CSV)
    assert "row_id" in psy_df.columns, "O CSV de features precisa conter a coluna row_id."

    feature_cols = [c for c in psy_df.columns if c != "row_id"]
    print("Total de features psicológicas:", len(feature_cols))

    # Exemplo: recuperar features do split de treino pelo row_id.
    train_ids = pd.DataFrame({"row_id": ds["train"]["row_id"]})
    train_features = train_ids.merge(psy_df, on="row_id", how="left")

    print(train_features.shape)
    display(train_features.head())
else:
    print("Nenhum CSV de features psicológicas informado. Pule esta célula por enquanto.")


Nenhum CSV de features psicológicas informado. Pule esta célula por enquanto.


## 13. Checklist de replicação

Use este checklist ao transformar este notebook de baseline em replicação completa:

- [ ] Rodar `MODE = "replication"` com todos os dados.
- [ ] Registrar GPU, tempo de treino e versões das bibliotecas.
- [ ] Salvar os splits (`split_row_ids.json`).
- [ ] Rodar pelo menos 3 seeds, se houver tempo.
- [ ] Extrair SEANCE, TAACO e TAALES para todos os textos.
- [ ] Fazer seleção por correlação de Pearson.
- [ ] Implementar o encoder psicológico sem positional encoding.
- [ ] Implementar fusão BERT + features psicológicas.
- [ ] Rodar ablações.
- [ ] Comparar com a Tabela 4 do artigo.
